In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):

    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

### Full DQN CartPole Training Code

In [3]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

In [4]:
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# hyperparameters
gamma = 0.99
learning_rate = 0.001
batch_size = 64
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
episodes = 500

# replay memory
memory = deque(maxlen=10000)


In [5]:
# neural network
model = DQN(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

In [6]:
def choose_action(state):
    if np.random.rand() < epsilon:
        return env.action_space.sample()

    state = torch.FloatTensor(state).unsqueeze(0)
    q_values = model(state)
    return torch.argmax(q_values).item()


In [7]:
def replay():

    if len(memory) < batch_size:
        return

    batch = random.sample(memory, batch_size)

    states, actions, rewards, next_states, dones = zip(*batch)

    states = torch.FloatTensor(states)
    next_states = torch.FloatTensor(next_states)
    actions = torch.LongTensor(actions)
    rewards = torch.FloatTensor(rewards)
    dones = torch.FloatTensor(dones)

    q_values = model(states)
    next_q_values = model(next_states)

    q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
    next_q_value = torch.max(next_q_values, 1)[0]

    expected_q = rewards + gamma * next_q_value * (1 - dones)

    loss = loss_fn(q_value, expected_q.detach())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [8]:
for episode in range(episodes):

    state, _ = env.reset()
    total_reward = 0

    done = False

    while not done:

        action = choose_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        memory.append((state, action, reward, next_state, done))

        state = next_state
        total_reward += reward

        replay()

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    print(f"Episode {episode}, Reward: {total_reward}")
env.close()

Episode 0, Reward: 20.0
Episode 1, Reward: 13.0
Episode 2, Reward: 16.0
Episode 3, Reward: 11.0
Episode 4, Reward: 13.0
Episode 5, Reward: 28.0
Episode 6, Reward: 26.0


/tmp/ipykernel_33157/1149990092.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  states = torch.FloatTensor(states)


Episode 7, Reward: 13.0
Episode 8, Reward: 14.0
Episode 9, Reward: 49.0
Episode 10, Reward: 26.0
Episode 11, Reward: 27.0
Episode 12, Reward: 25.0
Episode 13, Reward: 11.0
Episode 14, Reward: 10.0
Episode 15, Reward: 19.0
Episode 16, Reward: 18.0
Episode 17, Reward: 27.0
Episode 18, Reward: 13.0
Episode 19, Reward: 16.0
Episode 20, Reward: 16.0
Episode 21, Reward: 14.0
Episode 22, Reward: 22.0
Episode 23, Reward: 12.0
Episode 24, Reward: 41.0
Episode 25, Reward: 29.0
Episode 26, Reward: 23.0
Episode 27, Reward: 54.0
Episode 28, Reward: 60.0
Episode 29, Reward: 69.0
Episode 30, Reward: 19.0
Episode 31, Reward: 28.0
Episode 32, Reward: 20.0
Episode 33, Reward: 17.0
Episode 34, Reward: 28.0
Episode 35, Reward: 14.0
Episode 36, Reward: 24.0
Episode 37, Reward: 26.0
Episode 38, Reward: 16.0
Episode 39, Reward: 56.0
Episode 40, Reward: 46.0
Episode 41, Reward: 19.0
Episode 42, Reward: 43.0
Episode 43, Reward: 52.0
Episode 44, Reward: 23.0
Episode 45, Reward: 13.0
Episode 46, Reward: 42.0
Epi

In [9]:
torch.save(model.state_dict(), "cartpole_dqn.pth")
print("Model saved!")

Model saved!


### Run CartPole Using the Trained Model

In [11]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F


class DQN(nn.Module):

    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)


# create environment with rendering
env = gym.make("CartPole-v1", render_mode="human")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# load trained model
model = DQN(state_size, action_size)
model.load_state_dict(torch.load("cartpole_dqn.pth"))
model.eval()

state, _ = env.reset()

done = False

while not done:

    state_tensor = torch.FloatTensor(state).unsqueeze(0)

    with torch.no_grad():
        q_values = model(state_tensor)

    action = torch.argmax(q_values).item()

    state, reward, terminated, truncated, _ = env.step(action)

    done = terminated or truncated

env.close()